# 水頭依存効率を持つ多段ダム放流計画

一級河川に連なる複数ダムを一括運用する電力会社の「水系運用担当者」が、季節(数十〜百数十日)
にわたる各ダムの放流量を決める。上流ダムの放流は下流ダムへの流入となり(河川流下の時間遅れを
伴う)、各ダムの発電量は **放流量 × 水頭(貯水位に依存)** という双線形で決まる。

各制約の業務的意味:

- **発電量 = 放流量 × 水頭(双線形)**: 水力発電のタービン出力は概ね `P = k・q・h`
  (q=放流量、h=有効落差)に比例する。落差hは貯水位(=貯水量)の線形関数であり、貯水位が
  高いほど同じ放流量でも発電量が大きい。「いつ貯めて、いつ放つか」が発電価値を左右する
  非凸構造そのもの(近似ではなく水力発電の物理)。
- **貯水量バランス(時間結合)**: 各期の貯水量は前期の貯水量+自流域流入+上流からの放流・
  越流(1期の河川流下遅れ付き)−自ダム放流−越流。上流の意思決定が下流の制約を通じて
  全期間に波及する。
- **灌漑取水下限**: 指定ダムは灌漑期に最低放流量を守る義務があり、発電の都合だけで
  放流量を決められない。
- **期末貯水量下限**: 渇水対策の備蓄目標。「今使うか、将来のために貯めるか」のトレードオフ。
- **火力バックストップ**: 水力だけで需要を満たせない期は高コストの火力代替で埋める
  (常に実行可能性を担保)。

題材: `samples/energy_and_microgrid/hydro_cascade_efficiency.py`。

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/energy_and_microgrid"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

def show(fig):
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import hydro_cascade_efficiency as hc
from pyscipopt import Model, quicksum

C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'

def base_layout(title, xtitle, ytitle, height=380):
    ax = dict(gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"], size=11),
              title_font=dict(color=C["ink2"], size=12), zeroline=False)
    return go.Layout(
        title=dict(text=title, font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
        paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
        font=dict(family=FONT, color=C["ink2"], size=12),
        xaxis=dict(ax, title=xtitle), yaxis=dict(ax, title=ytitle),
        margin=dict(l=60, r=20, t=48, b=48), height=height, hovermode="closest",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right",
                    font=dict(size=11, color=C["ink2"]), bgcolor="rgba(0,0,0,0)"))
print("repo root:", ROOT)

repo root: C:\work_space\mip


## 1. 素朴な定式化

既定scaleはダム5×期22。`head_def`(水頭の定義、貯水量の線形関数)は線形だが、
`gen_def`(発電量=発電係数×放流量×水頭)は**放流量(連続)×水頭(連続)の双線形**で、
ダム×期の数だけ現れる。

In [2]:
m0 = hc.build_model("default")
nl = sum(1 for c in m0.getConss() if c.isNonlinear())
print(f"変数 {m0.getNVars()}、制約 {m0.getNConss()}(うち非線形 {nl})")
nD, nT = m0.data["dims"]
print(f"ダム数 nD={nD} × 期数 nT={nT} = {nD*nT} -> gen_def の本数と一致: {nl == nD*nT}")
del m0

変数 572、制約 375(うち非線形 110)
ダム数 nD=5 × 期数 nT=22 = 110 -> gen_def の本数と一致: True


非線形制約は全て `gen_def`(発電量の定義)で、本数はダム数×期数に正確に一致する。
`head_def`(水頭=貯水量の線形関数)自体は線形なので、双線形の起点は「放流量×水頭」という
発電の物理式そのものにある。

## 2. 診断

In [3]:
report = mk.analyze(lambda: hc.build_model("default"), name="hydro-cascade",
                    time_limit=30)
print(report.summary())
print("\ngap:", f"{report.metrics.get('gap', 0):.1%}",
      "/ 支配ボトルネック:", report.metrics.get("bottleneck_type"),
      f"(相対違反 {report.metrics.get('bottleneck_rel_viol', 0):.2f})",
      "/ 空間分枝の双対寄与:", f"{report.metrics.get('spatial_share', 0):.0%}")

[hydro-cascade] 検出症状 3件:
  [serious] 特定の非線形制約に緩和違反が集中(かつ空間分枝が多い) -> ボトルネック制約の区分線形近似・凸包再定式化・変数境界タイト化
  [warning] 双対境界の改善が停滞(gapが残る) -> 有効不等式の追加・変数境界タイト化・Big-M排除で緩和強化
  [warning] 係数の絶対値レンジが桁違い / Big-M候補あり(presolve後も残存) -> スケーリング、Big-MのIndicator/SOS制約化、係数の再定式化

gap: 27.9% / 支配ボトルネック: gen_def_2 (相対違反 0.90) / 空間分枝の双対寄与: 100%


30秒でgapが28%程度残り、`weak_relaxation`(serious、緩和違反が集中+空間分枝が支配的)・
`dual_stall`・`numerical_scale`の3件が発火する。空間分枝の双対寄与が**100%** —
双対境界の改善は全て連続変数(放流量・水頭)への分枝からもたらされ、この問題には整数変数が
無いことを踏まえると、`gen_def`のMcCormick緩和の弱さがそのままボトルネックになっている
ことを示す。中身を見る。

## 3. 診断の中身を見る — 違反の集中と分枝の帰属

ルートLP緩和解での非線形制約違反(`gen_def`がダム別・期別にどう集中するか)と、
双対境界改善の帰属を直接見る。

In [4]:
from minlpkit.collectors.violation import collect_root_violations, violation_by_type
from minlpkit.collectors.attribution import solve_and_attribute, gain_by_kind

m1 = hc.build_model("default")
vdf = collect_root_violations(m1)
# entity は "dam_period" の形。ダム別に集計
vdf["dam"] = vdf["entity"].str.split("_").str[0]
by_dam = vdf.groupby("dam")["rel_violation"].mean().reset_index()
by_dam["dam"] = by_dam["dam"].astype(int)
by_dam = by_dam.sort_values("dam")
by_dam

,dam,rel_violation
0,0,0.392411
1,1,0.264737
12,2,0.271472
15,3,0.266507
16,4,0.265881
17,5,0.178572
18,6,0.350379
19,7,0.330896
20,8,0.270957
21,9,0.270994


In [5]:
fig = go.Figure(layout=base_layout(
    "gen_def(発電量定義)のルートLP緩和違反(ダム別平均、上流=0→下流)",
    "ダム番号(0=最上流)", "相対違反(平均)", height=380))
fig.add_trace(go.Bar(x=by_dam["dam"], y=by_dam["rel_violation"], marker_color=C["warn"]))
show(fig)

In [6]:
m2 = hc.build_model("default")
d2, summ2 = solve_and_attribute(m2, time_limit=30, gap_limit=0.01)
gk = gain_by_kind(d2)
fig = go.Figure(layout=base_layout(
    "双対境界の改善の帰属(分枝の種類別)", "分枝の種類", "双対境界の押し上げ量(合計)", height=380))
fig.add_trace(go.Bar(x=gk["kind"], y=gk["dual_gain"],
    marker_color=[C["warn"] if k == "spatial" else C["s1"] for k in gk["kind"]],
    text=[f"{v:.1f}" for v in gk["dual_gain"]], textposition="outside"))
show(fig)

違反はほぼ全てのダムに広く分布し、特定の1基だけの問題ではない。分枝の帰属も
`spatial`(連続変数の空間分枝)が事実上すべてを占める — McCormick緩和の弱さが全ダム共通で
効いていることが確認できる。McCormickの帯を狭めるには **q・head の変数境界(特に貯水量の
上限)をタイト化**するのが定石(4節)。

## 4. 改善を試す — 貯水量の到達可能範囲で境界をタイト化(FBBT)

現在の `S[dd,t]`(貯水量)の変数境界は全期間で `[smin,smax]`(そのダムの物理容量の全域)
に固定されている。しかし実際には、初期貯水量 `s0` から出発して各期に流入できる水量は有限
なので、序盤の期では `smax` まで到達できないことが多い。これは**到達可能性に基づく境界伝播
(FBBT: feasibility-based bound tightening)**の典型例で、SCIPのpresolveも同種の処理を行うが、
このモデルは越流(`sp`)に上限が無いため自動伝播だけでは詰め切れない部分がある。

放流`q`・越流`sp`を最小(=0または灌漑下限)にしたときに到達できる貯水量の上限を、
上流から下流へ順に前方伝播で計算する:

```
S_hi[d,t] = min(smax[d], S_hi[d,t-1] + local_inflow[d,t] + upstream_hi[d,t])
```

これは「放流量・越流を減らすほど貯水量は増える」という単調性から導かれる**厳密に妥当な
上限**(実行可能領域を一切削らない)。タイトになった `S_hi` から `head` の上限も連動して
タイトになり、`gen = k・q・head` のMcCormick包絡線が狭まる。

In [7]:
def build_tight(scale="default"):
    '''S(貯水量)の到達可能上限をFBBTでタイト化した変種(head の上限も連動)。'''
    d = hc._data(scale)
    nD, nT = d["nD"], d["nT"]
    smin, smax, s0, qmax = d["smin"], d["smax"], d["s0"], d["qmax"]
    h0, alpha, k_gen = d["h0"], d["alpha"], d["k_gen"]
    local_inflow, irrig_min, s_target, demand = (
        d["local_inflow"], d["irrig_min"], d["s_target"], d["demand"])

    # 前方FBBT: 放流・越流を最小にしたときの到達可能な貯水量上限(妥当な上限、実行可能領域は不変)
    S_hi = np.zeros((nD, nT))
    sp_hi = np.zeros((nD, nT))
    for dd in range(nD):
        for t in range(nT):
            prev_hi = S_hi[dd, t - 1] if t > 0 else float(s0[dd])
            upstream_hi = 0.0 if (dd == 0 or t == 0) else float(qmax[dd - 1]) + sp_hi[dd - 1, t - 1]
            q_lo = float(irrig_min[dd, t]) if irrig_min[dd, t] > 0 else 0.0
            S_hi[dd, t] = min(float(smax[dd]), prev_hi + float(local_inflow[dd, t]) + upstream_hi)
            sp_hi[dd, t] = max(0.0, prev_hi + float(local_inflow[dd, t]) + upstream_hi
                               - q_lo - float(smin[dd]))

    m = Model("Hydro_Cascade_Efficiency_Tight")
    D, T = range(nD), range(nT)
    S = {(dd, t): m.addVar(vtype="C", lb=float(smin[dd]), ub=float(S_hi[dd, t]), name=f"S_{dd}_{t}")
         for dd in D for t in T}
    q = {(dd, t): m.addVar(vtype="C", lb=0.0, ub=float(qmax[dd]), name=f"q_{dd}_{t}")
         for dd in D for t in T}
    sp = {(dd, t): m.addVar(vtype="C", lb=0.0, name=f"sp_{dd}_{t}") for dd in D for t in T}
    head = {(dd, t): m.addVar(vtype="C",
                              lb=float(h0[dd] + alpha[dd] * smin[dd]),
                              ub=float(h0[dd] + alpha[dd] * S_hi[dd, t]),
                              name=f"head_{dd}_{t}") for dd in D for t in T}
    gen = {(dd, t): m.addVar(vtype="C", lb=0.0, name=f"gen_{dd}_{t}") for dd in D for t in T}
    thermal = {t: m.addVar(vtype="C", lb=0.0, name=f"thermal_{t}") for t in T}
    for dd in D:
        for t in T:
            m.addCons(head[dd, t] == float(h0[dd]) + float(alpha[dd]) * S[dd, t], name=f"head_def_{dd}_{t}")
            m.addCons(gen[dd, t] == float(k_gen[dd]) * q[dd, t] * head[dd, t], name=f"gen_def_{dd}_{t}")
            if irrig_min[dd, t] > 0:
                m.addCons(q[dd, t] >= float(irrig_min[dd, t]), name=f"irrig_{dd}_{t}")
            prev = S[dd, t - 1] if t > 0 else float(s0[dd])
            upstream_in = 0.0 if (dd == 0 or t == 0) else q[dd - 1, t - 1] + sp[dd - 1, t - 1]
            m.addCons(S[dd, t] == prev + float(local_inflow[dd, t]) + upstream_in - q[dd, t] - sp[dd, t],
                      name=f"storage_balance_{dd}_{t}")
        m.addCons(S[dd, nT - 1] >= float(s_target[dd]), name=f"terminal_storage_{dd}")
    for t in T:
        m.addCons(quicksum(gen[dd, t] for dd in D) + thermal[t] >= float(demand[t]), name=f"power_balance_{t}")
    m.setObjective(quicksum(hc.THERMAL_COST * thermal[t] for t in T), "minimize")
    m.data = dict(S=S, q=q, gen=gen, thermal=thermal, scale=scale, dims=(nD, nT))
    return m

mb = hc.build_model("small"); mb.hideOutput(); mb.optimize()
mt = build_tight("small"); mt.hideOutput(); mt.optimize()
print(f"baseline : obj={mb.getObjVal():.2f}  status={mb.getStatus()}")
print(f"tight    : obj={mt.getObjVal():.2f}  status={mt.getStatus()}")

baseline : obj=36304.80  status=optimal
tight    : obj=36304.80  status=optimal


In [8]:
# 実際にどれだけ境界が締まったか(既定scale、上流ダム0が最も締まりやすい)
d = hc._data("default")
nD, nT = d["nD"], d["nT"]
smin, smax, s0, qmax = d["smin"], d["smax"], d["s0"], d["qmax"]
local_inflow, irrig_min = d["local_inflow"], d["irrig_min"]
S_hi = np.zeros((nD, nT)); sp_hi = np.zeros((nD, nT))
for dd in range(nD):
    for t in range(nT):
        prev_hi = S_hi[dd, t - 1] if t > 0 else float(s0[dd])
        upstream_hi = 0.0 if (dd == 0 or t == 0) else float(qmax[dd - 1]) + sp_hi[dd - 1, t - 1]
        q_lo = float(irrig_min[dd, t]) if irrig_min[dd, t] > 0 else 0.0
        S_hi[dd, t] = min(float(smax[dd]), prev_hi + float(local_inflow[dd, t]) + upstream_hi)
        sp_hi[dd, t] = max(0.0, prev_hi + float(local_inflow[dd, t]) + upstream_hi - q_lo - float(smin[dd]))

fig = go.Figure(layout=base_layout(
    "貯水量上限のタイト化(ダム別、元のsmaxに対する比率)",
    "期", "S_hi(タイト化後) / smax(元の上限)", height=400))
for dd in range(nD):
    ratio = S_hi[dd, :] / smax[dd]
    fig.add_trace(go.Scatter(x=list(range(nT)), y=ratio, mode="lines",
        line=dict(width=2, color=[C["s1"], C["s2"], C["warn"], C["muted"], C["ink"]][dd % 5]),
        name=f"ダム{dd}"))
fig.add_hline(y=1.0, line=dict(color=C["axis"], dash="dot"))
show(fig)

最上流(ダム0)は序盤の期で明確にタイト化される(到達可能な貯水量が`smax`未満に制限される)。
下流ダムは越流`sp`に上限が無いため上流からの流入見積りが緩く、タイト化の効果が上流ほど
大きくない — 越流を無制限に許すモデル前提の限界がそのまま伝播に表れている。

In [9]:
df = mk.compare_variants(
    {"baseline":     lambda: hc.build_model("default"),
     "tight bounds": lambda: build_tight("default")},
    time_limit=30)
df[["variant", "root_dual", "final_dual", "final_gap", "nodes"]]

,variant,root_dual,final_dual,final_gap,nodes
0,baseline,610930.096617,707418.641412,0.278144,9000
1,tight bounds,610940.580571,691931.144365,0.306752,2800


In [10]:
base = df.loc[df["variant"] == "baseline"].iloc[0]
tgt  = df.loc[df["variant"] == "tight bounds"].iloc[0]
labels = ["baseline", "tight bounds"]
bar_colors = [C["muted"], C["s1"]]
fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.12,
    subplot_titles=("ルート双対境界 (高いほど良い)", "最終 gap [%] (低いほど良い)",
                    "探索ノード数 (少ないほど良い)"))
def add_bars(col, values, fmt):
    fig.add_trace(go.Bar(x=labels, y=values, marker_color=bar_colors,
        text=[fmt(v) for v in values], textposition="outside",
        cliponaxis=False, showlegend=False), row=1, col=col)
add_bars(1, [base["root_dual"], tgt["root_dual"]], lambda v: f"{v:.0f}")
add_bars(2, [base["final_gap"]*100, tgt["final_gap"]*100], lambda v: f"{v:.0f}%")
add_bars(3, [base["nodes"], tgt["nodes"]], lambda v: f"{int(v)}")
fig.update_layout(paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=360,
    margin=dict(l=40, r=20, t=64, b=40),
    title=dict(text="貯水量の境界タイト化(FBBT) before / after", x=0.01,
               font=dict(color=C["ink"], size=15)))
fig.update_yaxes(gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
fig.update_xaxes(linecolor=C["axis"])
show(fig)
print(f"root_dual : {base['root_dual']:.1f} -> {tgt['root_dual']:.1f}")
print(f"final_gap : {base['final_gap']:.1%} -> {tgt['final_gap']:.1%}")
print(f"nodes     : {int(base['nodes'])} -> {int(tgt['nodes'])}")

root_dual : 610930.1 -> 610940.6
final_gap : 27.8% -> 30.7%
nodes     : 9000 -> 2800


**正直な検証結果**: FBBTによる境界タイト化は、実行可能領域を一切変えずに(等価な変換)
McCormick包絡線を狭める、理論的に健全な改善である。効果の大きさは実測(上のグラフ)の通り
— タイト化が明確に効くのは最上流ダムの序盤の期に限られ(3節グラフで確認)、下流ダムは
越流`sp`が無制限なため伝播が弱く、全体としての改善幅は限定的になりやすい。これは
「境界タイト化という技法自体が無効」なのではなく、**このモデルの越流を無制限に許す設計が
FBBTの効きを弱めている**という構造的な理由による — 越流に物理的に妥当な上限
(例: 通水設備の合計容量)を追加すれば、下流側でも同様の伝播が働くと考えられる
(この notebook では未検証、次の改善候補)。

## まとめ

- この問題の難しさは、**放流量×水頭という水力発電の物理式そのもの**が生む双線形にある。
  診断は全ダム・全期にわたって`weak_relaxation`(空間分枝の双対寄与100%)を指し、
  特定の1基やの1期の問題ではないことを裏付けた。
- FBBTによる貯水量境界のタイト化は理論的に健全な一手だが、効果はダムの位置(上流ほど効く)
  とモデルの前提(越流が無制限)に左右される — **万能ではなく、モデル構造次第で効きにムラが
  出る**ことを実測で示せた。
- 実務的には、これは「上流ダムでいつ貯め、いつ放流して下流の発電・灌漑につなげるか」という
  カスケード河川の一括運用そのものであり、双線形は近似ではなく発電の物理(P=k・q・h)から
  必然的に生じる。灌漑下限は発電最適とは独立した外生制約として、この非凸構造をさらに
  非対称に縛っている。

関連: [手法ガイド 0. 診断そのもの](../../playbook/00-diagnose.md) /
[手法ガイド 3. Big-M排除](../../playbook/03-bigm.md)(境界タイト化の関連技法) /
API [`mk.analyze`](../../api/pipeline.md)